# Experiment 3 — Short-term Load Response

**Research question:** Can we predict which players are at risk of a status deterioration tomorrow? And does training intensity causally affect this risk?

**Target:** `Status Decrease` (binary, ~3–5% prevalence)
**Treatment (causal framing):** `Training Intensity Yesterday` — the training load prescribed by the coaching staff
**Covariates:** Morning wellness, ACWR, schedule position, player profile
**Models:** Logistic Regression, XGBoost (GPU), CatBoost (GPU), TabPFN (GPU)

---

### Table of Contents

| Section | Content |
|---------|---------|
| **0. Imports** | Libraries, project paths, experiment runner |
| **Configuration** | Lags, covariates, models, modes, GPU overrides |
| **1. Run experiments** | All mode × model × lag combinations |
| **2. Comparison table** | ROC-AUC, precision, recall, F1 for every configuration |
| **3. ROC AUC visualisation** | Bar charts across models, lags, and modes |
| **4. Lag effect** | Does more history improve predictions? |
| **5. ROC and PR curves** | Best prediction-mode model |
| **6. Causal framing** | Logistic regression coefficients — confounding by indication |
| **7. Per-player breakdown** | AUC per player for the best configuration |
| **8. Summary** | Raw results table (no interpretations) |



---
## CONFIGURATION — edit this cell to change the experiment

In [ ]:
# ============================================================
#  EXPERIMENT 3 CONFIGURATION
# ============================================================

# --- Lag values to compare -------------------------------------------
LAGS = [0, 1, 2, 3]

# --- Train / test / validation split ---------------------------------
TEST_SIZE = 0.2
VAL_SIZE  = 0.1

# --- Predictor columns -----------------------------------------------
# Morning-assessment covariates (available before the session).
# 'Training Intensity Yesterday' = yesterday's session intensity,
# fully known at morning assessment time (it is day t-1 data).
#
# Excluded compared to earlier iterations:
#   - 'Activity Type Yesterday': session-type label from yesterday;
#     too colinear with Training Intensity Yesterday (both describe the
#     same session from different angles) and adds categorical noise.
#   - 'Medical Availability Last 14 Days': a rolling availability
#     percentage — already partially captured by Status and ACWR-based
#     features; excluded to keep the covariate set focused on
#     physiological state rather than administrative records.
COVARIATES = [
    # Morning wellness composites
    'Physical State',                        # Morning composite: fatigue/soreness/readiness
    'Mental State',                          # Morning composite: mood/stress/sleep
    # Yesterday's load -- ACWR ratios
    'Total Distance (ACWR) Yesterday',       # Workload: ACWR total distance
    'High Speed Distance (ACWR) Yesterday',  # Workload: ACWR high-speed distance
    'Any ACWR Danger',                       # Binary: any ACWR > 1.5
    # Yesterday's load -- GPS benchmark %
    'Total Distance % Yesterday',            # GPS total distance as % of match benchmark
    'High Speed Distance % Yesterday',       # GPS high-speed distance as % of match benchmark
    # Yesterday's load -- composite + subjective
    'Training Intensity Yesterday',          # KEY: treatment variable in causal_framing mode
    'Perceived Exertion Yesterday',          # RPE z-score
    # Yesterday's raw GPS/HR
    'Total Minutes Yesterday',               # Raw session volume
    'Avg Heart Rate Yesterday',              # Average heart rate
    'Heart Rate Exertion Yesterday',         # Heart rate exertion index
    # Temporal context
    'Days Since Game', 'Days Until Match',   # Schedule position
    # Player profile (NO Player ID -- not useful for status prediction)
    'Position',                              # Playing position
]

# --- Models to run ---------------------------------------------------
MODELS = ['log_reg', 'xgboost', 'catboost', 'tabpfn']

# --- Modes to run ----------------------------------------------------
MODES = ['prediction', 'causal_framing']

# ============================================================
print(f"Lags to compare:  {LAGS}")
print(f"Modes:            {MODES}")
print(f"Models:           {MODELS}")
print(f"Covariates:       {len(COVARIATES)}")
print(f"Test / Val split: {TEST_SIZE} / {VAL_SIZE}")
# --- GPU overrides -------------------------------------------------------
GPU_OVERRIDES = {
    'xgboost':  {'device': 'cuda'},
    'catboost': {'task_type': 'GPU'},
    'tabpfn':   {'device': 'cuda', 'ignore_pretraining_limits': True},
}




## 0. Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.extend([str(ROOT), str(ROOT / 'src'), str(ROOT / 'scripts')])

from Experiment3 import run_experiment

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print("Setup complete.")

## 1. Run all models × lags × modes

Results stored in `all_results[mode][model_type][lag]`.

In [ ]:
all_results = {mode: {m: {} for m in MODELS} for mode in MODES}

for mode in MODES:
    for model_type in MODELS:
        gpu_kwargs = GPU_OVERRIDES.get(model_type, {})
        for lag in LAGS:
            print(f"\n>>> {mode}  {model_type}  lag={lag}")
            all_results[mode][model_type][lag] = run_experiment(
                covariates=COVARIATES,
                lag=lag,
                model_type=model_type,
                mode=mode,
                test_size=TEST_SIZE,
                val_size=VAL_SIZE,
                verbose=True,
                **gpu_kwargs,
            )

print("\nAll experiments complete.")



## 2. Comparison table

In [ ]:
rows = []
for mode in MODES:
    for model_type in MODELS:
        for lag in LAGS:
            m = all_results[mode][model_type][lag]['metrics']
            rows.append({
                'Mode':          mode,
                'Model':         model_type,
                'Lag':           lag,
                'ROC AUC':       round(m['roc_auc'],       4),
                'Avg Prec':      round(m['avg_precision'],  4),
                'F1':            round(m['f1'],             4),
                'Recall':        round(m['recall'],         4),
                'Precision':     round(m['precision'],      4),
                'Prevalence':    round(m['prevalence'],     3),
                'N test':        m['n_test'],
                'N pos':         m['n_positive_test'],
            })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

**Reading the table:**
- **ROC AUC 0.5** = random; higher is better (not affected by class imbalance).
- **Avg Precision** = area under the PR curve; more sensitive to class imbalance than ROC AUC.
- **Comparison across modes**: if `causal_framing` AUC > `prediction` AUC, same-day load has strong contemporaneous signal; if similar, morning state dominates.

## 3. ROC AUC across models, lags, modes

In [ ]:
LAG_COLORS = {1: '#2c7bb6', 2: '#d7191c', 3: '#1a9641'}
n_lags  = len(LAGS)
n_modes = len(MODES)

fig, axes = plt.subplots(1, n_modes, figsize=(7 * n_modes, 4), sharey=True)
if n_modes == 1:
    axes = [axes]

x = np.arange(len(MODELS))
w = 0.22

for ax, mode in zip(axes, MODES):
    for i, lag in enumerate(LAGS):
        offset = (i - (n_lags - 1) / 2) * w
        aucs = [all_results[mode][m][lag]['metrics']['roc_auc'] for m in MODELS]
        ax.bar(x + offset, aucs, w, label=f'lag={lag}', color=LAG_COLORS[lag])
    ax.axhline(0.5, color='grey', ls='--', lw=1, label='random')
    ax.set_xticks(x); ax.set_xticklabels(MODELS)
    ax.set_ylabel('ROC AUC'); ax.set_title(f'mode={mode}')
    ax.legend(fontsize=8)

fig.suptitle('Experiment 3: ROC AUC by Mode, Model, and Lag', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Lag effect

In [ ]:
fig, axes = plt.subplots(1, n_modes, figsize=(7 * n_modes, 4), sharey=True)
if n_modes == 1:
    axes = [axes]

markers = ['o', 's', '^']

for ax, mode in zip(axes, MODES):
    for j, model_type in enumerate(MODELS):
        aucs = [all_results[mode][model_type][lag]['metrics']['roc_auc']
                for lag in LAGS]
        ax.plot(LAGS, aucs, marker=markers[j % len(markers)],
                label=model_type, lw=2)
    ax.axhline(0.5, color='grey', ls='--', lw=1)
    ax.set_xlabel('Lag'); ax.set_ylabel('ROC AUC')
    ax.set_title(f'AUC vs Lag — mode={mode}')
    ax.set_xticks(LAGS)
    ax.legend(fontsize=8)

fig.suptitle('Experiment 3: Does More History Help?', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. ROC and PR curves — best prediction-mode model

In [ ]:
# Best (model, lag) in prediction mode
best_pred = max(
    [(m, l) for m in MODELS for l in LAGS],
    key=lambda ml: all_results['prediction'][ml[0]][ml[1]]['metrics']['roc_auc']
)
bm, bl = best_pred
res = all_results['prediction'][bm][bl]
print(f"Best prediction mode: {bm} lag={bl}  "
      f"AUC={res['metrics']['roc_auc']:.4f}  "
      f"AP={res['metrics']['avg_precision']:.4f}")

y_true = res['y_test_true']
y_prob = res['y_test_pred']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fpr, tpr, _ = roc_curve(y_true, y_prob)
ax = axes[0]
ax.plot(fpr, tpr, color='#2c7bb6', lw=2,
        label=f"AUC = {res['metrics']['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — {bm} lag={bl} (prediction)')
ax.legend()

prec, rec, _ = precision_recall_curve(y_true, y_prob)
ax = axes[1]
ax.plot(rec, prec, color='#d7191c', lw=2,
        label=f"AP = {res['metrics']['avg_precision']:.3f}")
ax.axhline(res['metrics']['prevalence'], color='grey', ls='--', lw=1,
           label=f"Prevalence = {res['metrics']['prevalence']:.3f}")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall — {bm} lag={bl} (prediction)')
ax.legend(fontsize=8)

plt.suptitle('Prediction mode — best model', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Causal framing — logistic regression coefficients

In `causal_framing` mode (`target_horizon=0`), we predict **today's** Status Decrease from this morning's covariates including `Training Intensity Yesterday` ($A_{t-1}$).

**What the coefficient on `Training Intensity Yesterday` tells us:**

The raw log-odds coefficient is a **biased observational association**, not a causal effect.  Due to confounding by indication, we expect it to be near zero or negative (coaches give hard sessions to fresh players who are less likely to deteriorate).  A negative coefficient is therefore the expected sign — confirming the confounding structure rather than showing a protective effect of hard training.

**How to use this correctly:**  
- The full model $Q(A_{t-1}, L_t)$ from this mode is the **outcome model** for G-computation  
- Combine with the propensity model $\hat{\pi}(A_t \mid L_t)$ from Experiment 2 to estimate the Average Treatment Effect via IPTW: $\hat{E}[Y^a] = \frac{1}{n}\sum_i \frac{\mathbf{1}[A_i = a]}{\hat{\pi}(A_i \mid L_i)} Y_i$

In [ ]:
# Use the best lag for logistic regression in causal_framing mode
best_lag_cf = max(
    LAGS,
    key=lambda l: all_results['causal_framing']['log_reg'][l]['metrics']['roc_auc']
)
res_cf = all_results['causal_framing']['log_reg'][best_lag_cf]
print(f"Logistic regression causal_framing  lag={best_lag_cf}  "
      f"AUC={res_cf['metrics']['roc_auc']:.4f}")

weights_cf = res_cf.get('model_weights', {})
fnames_cf  = res_cf.get('feature_names', [])
coefs      = weights_cf.get('coefficients')

if coefs is not None and len(fnames_cf) > 0:
    # LogisticRegression.coef_ for binary classification has shape (1, n_features).
    # .tolist() therefore returns [[...]] — flatten to a 1-D array before use.
    coefs_flat = np.array(coefs).flatten()

    if len(coefs_flat) != len(fnames_cf):
        print(f"WARNING: length mismatch — {len(fnames_cf)} feature names "
              f"vs {len(coefs_flat)} coefficients. Skipping plot.")
    else:
        coef_df = pd.DataFrame({'feature': fnames_cf, 'coef': coefs_flat})
        coef_df = coef_df.reindex(
            coef_df['coef'].abs().sort_values(ascending=False).index
        ).head(25)

        is_ti = coef_df['feature'].str.contains('Training Intensity', case=False)
        bar_colors = ['#d7191c' if t else '#2c7bb6' for t in is_ti]

        fig, ax = plt.subplots(figsize=(9, 7))
        ax.barh(coef_df['feature'][::-1], coef_df['coef'][::-1],
                color=bar_colors[::-1])
        ax.axvline(0, color='black', lw=0.8)
        ax.set_xlabel('Log-odds coefficient')
        ax.set_title(
            f'Logistic coefficients (causal_framing, lag={best_lag_cf})\n'
            'Red = Training Intensity Yesterday (treatment proxy)'
        )
        plt.tight_layout()
        plt.show()

        ti_rows = coef_df[coef_df['feature'].str.contains('Training Intensity', case=False)]
        if not ti_rows.empty:
            print("Training Intensity Yesterday coefficient(s):")
            print(ti_rows.to_string(index=False))
            print()
            print("CAUTION: This is a biased observational association.")
            print("Causal estimation requires G-computation or IPTW (see Exp 2 propensity model).")
else:
    print("No coefficients available.")

## 7. Per-player breakdown — best prediction model

In [ ]:
res = all_results['prediction'][bm][bl]
pp  = res.get('per_player', {})

if pp:
    pp_df = pd.DataFrame([
        {'Player ID': pid, 'N': v['n'], 'N pos': v['n_positive'],
         'ROC AUC': round(v['roc_auc'], 4) if v['roc_auc'] is not None else float('nan'),
         'F1': round(v['f1'], 4)}
        for pid, v in sorted(pp.items())
    ])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.bar(pp_df['Player ID'].astype(str), pp_df['ROC AUC'].fillna(0.5),
           color='#2c7bb6')
    ax.axhline(0.5, color='grey', ls='--', lw=1)
    ax.set_xlabel('Player ID'); ax.set_ylabel('ROC AUC')
    ax.set_title('Per-player ROC AUC (prediction mode)')
    ax.tick_params(axis='x', rotation=90)

    ax = axes[1]
    ax.bar(pp_df['Player ID'].astype(str), pp_df['N pos'], color='#d7191c')
    ax.set_xlabel('Player ID'); ax.set_ylabel('N status-decrease events')
    ax.set_title('Positive events per player (test set)')
    ax.tick_params(axis='x', rotation=90)

    plt.suptitle(f'{bm} lag={bl} — Per-player', fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(pp_df.to_string(index=False))
    print("\nNote: AUC is NaN for players with zero positive events in the test set.")
else:
    print("No per-player breakdown available.")

## 8. Summary

All experimental findings printed in plain text — scroll through the output of the next cell for the complete results report.

In [ ]:
try:
    print("=" * 72)
    print("EXPERIMENT 3 -- RESULTS SUMMARY")
    print("=" * 72)
    print()
    for mode in MODES:
        print(f"MODE: {mode}")
        print("-" * 72)
        print(f"  {'Model':<12} {'Lag':>4} {'AUC':>8} {'AvgPrec':>8} {'F1':>6} {'Prec':>6} {'Rec':>6} {'Acc':>6}")
        for model_type in MODELS:
            for lag in LAGS:
                r = all_results[mode][model_type].get(lag)
                if r is None: continue
                m = r['metrics']
                auc = m.get('roc_auc', 0) or 0
                ap = m.get('avg_precision', 0) or 0
                f1 = m.get('f1', 0) or 0
                prec = m.get('precision', 0) or 0
                rec = m.get('recall', 0) or 0
                acc = m.get('accuracy', 0) or 0
                print(f"  {model_type:<12} {lag:>4} {auc:>8.4f} {ap:>8.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f} {acc:>6.4f}")
        print()
    # LogReg coefficients (causal_framing)
    print("LOGISTIC REGRESSION COEFFICIENTS (causal_framing)")
    print("-" * 72)
    for lag in LAGS:
        lr = all_results.get('causal_framing', {}).get('log_reg', {}).get(lag)
        if lr is None: continue
        w = lr.get('model_weights', {})
        coefs = w.get('coefficients')
        fn = lr.get('feature_names', [])
        if coefs is not None and len(fn) > 0:
            ca = np.array(coefs).flatten()
            if len(ca) == len(fn):
                print(f"  Lag={lag}:")
                cdf = pd.DataFrame({'feature': fn, 'coef': ca})
                cdf = cdf.reindex(cdf['coef'].abs().sort_values(ascending=False).index)
                print(cdf.head(15).to_string(index=False))
                print()
    print("=" * 72)
except NameError:
    print("[Summary] Run experiment cells first.")

